In [30]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from pypdf import PdfReader
import glob

In [39]:
# Setting up
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"
db_name='vector_db'
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [25]:
knowledge_base_path = "GP_Knowledge_Base/**/*.pdf"
files = glob.glob(knowledge_base_path, recursive=True)

print(f"Found {len(files)} files in the knowledge base")
print(files)

entire_knowledge_base = ""

for file_path in files:
    reader = PdfReader(file_path)
    
    for page in reader.pages:
        text = page.extract_text()
        if text:  # avoid None
            entire_knowledge_base += text
    
    entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)


Found 2 files in the knowledge base
['GP_Knowledge_Base/CV/George Penny - CV.pdf', 'GP_Knowledge_Base/CV/Linkedin_Profile.pdf']
Total characters in knowledge base: 7,693


In [26]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 1,580


In [32]:
#### Load all the PDF's
folders = glob.glob("GP_Knowledge_Base/*")

documents = []

for folder in folders:
    if not os.path.isdir(folder):
        continue

    doc_type = os.path.basename(folder)
    pdf_files = glob.glob(os.path.join(folder, "**", "*.pdf"), recursive=True)

    for file_path in pdf_files:
        loader = PyPDFLoader(file_path)
        file_docs = loader.load()

        for doc in file_docs:
            doc.metadata["doc_type"] = doc_type
            doc.metadata["source_file"] = os.path.basename(file_path)
            documents.append(doc)

print(f"Loaded {len(documents)} documents")
print(f"{documents}")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)


Loaded 4 documents
[Document(metadata={'producer': 'macOS Version 13.6.6 (Build 22G630) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20241205150245Z00'00'", 'moddate': "D:20241205150245Z00'00'", 'source': 'GP_Knowledge_Base/CV/George Penny - CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'doc_type': 'CV', 'source_file': 'George Penny - CV.pdf'}, page_content='George Penny – Curriculum Vitae  Name: George Scott Penny Email: george.penny@live.com    Mobile: +447894 240 738  LinkedIn: https://www.linkedin.com/in/george-penny-11309590/  Profile I am a highly motivated and results-oriented professional with a strong analytical background and a proven ability to deliver impactful solutions in risk management, operations, and credit analytics. Known for approaching challenges with a positive and methodical mindset, I have consistently demonstrated leadership, innovation, and collaboration, earning multiple "Employee of the Month" nominations throughout my career.   Pro

In [ ]:
## chunking - small file so may not be required - doing it for the fun!!

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 11 chunks
First chunk:

page_content='George Penny – Curriculum Vitae  Name: George Scott Penny Email: george.penny@live.com    Mobile: +447894 240 738  LinkedIn: https://www.linkedin.com/in/george-penny-11309590/  Profile I am a highly motivated and results-oriented professional with a strong analytical background and a proven ability to deliver impactful solutions in risk management, operations, and credit analytics. Known for approaching challenges with a positive and methodical mindset, I have consistently demonstrated leadership, innovation, and collaboration, earning multiple "Employee of the Month" nominations throughout my career.   Professional Experience  Stenn | Risk and Operations Lead June 2024 - Present  A leader within Stenn’s product and tech domain, reporting directly to the VP of Data. I played a pivotal role in scaling the risk and operations strategy, driving automation, and influencing cross-functional stakeholders. As part of the risk and operations l

In [40]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created with 11 documents


In [41]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 11 vectors with 384 dimensions in the vector store


In [45]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange','Purple'][['products', 'employees', 'contracts', 'company','CV'].index(t)] for t in doc_types]

In [48]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2,perplexity=5, random_state=11)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [49]:

tsne = TSNE(n_components=3, perplexity=5, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

In [50]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [52]:
retriever.invoke("Who is George?")

[Document(id='b91b2f5a-3b5f-431d-953c-a31f76d957fe', metadata={'page_label': '1', 'page': 0, 'producer': 'macOS Version 13.6.6 (Build 22G630) Quartz PDFContext', 'creationdate': "D:20241205150245Z00'00'", 'doc_type': 'CV', 'creator': 'PyPDF', 'source': 'GP_Knowledge_Base/CV/George Penny - CV.pdf', 'source_file': 'George Penny - CV.pdf', 'total_pages': 2, 'moddate': "D:20241205150245Z00'00'"}, page_content='George Penny – Curriculum Vitae  Name: George Scott Penny Email: george.penny@live.com    Mobile: +447894 240 738  LinkedIn: https://www.linkedin.com/in/george-penny-11309590/  Profile I am a highly motivated and results-oriented professional with a strong analytical background and a proven ability to deliver impactful solutions in risk management, operations, and credit analytics. Known for approaching challenges with a positive and methodical mindset, I have consistently demonstrated leadership, innovation, and collaboration, earning multiple "Employee of the Month" nominations thr

In [53]:
llm.invoke("Who is George?")

AIMessage(content='Could you please provide more context or specify which George you are referring to? There are many individuals named George, and I’d be happy to help with more details.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 11, 'total_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_4e06acaf45', 'id': 'chatcmpl-DOUtl1n10s6nz81dkN0hz0t52uGO5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d3636-3a00-7373-827f-ebbe1c0db4c3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 33, 'total_tokens': 44, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_det

In [54]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly talent aquisition manager representing George Penny.
You are chatting with a user about George Penny.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [55]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("What is impressive about George Peny?", [])